# Logging my journey learning LLMs and PyTorch

It is long overdue to start learning PyTorch, and there is no better way to do it than by building a toy LLM. I've gone through a list of papers that helped me to understand how LLMs come to be, and I think it's a good idea to build a toy LLM to better connect the theory with practice.

# Starting the local colab docker image

Use the following command:
```
docker run -it --gpus=all -p 127.0.0.1:9000:8080 --mount type=bind,src="${HOME}/colab/checkpoints",dst=/content/checkpoints us-docker.pkg.dev/colab-images/public/runtime
```

In [1]:
!pwd && ls

/content
checkpoints  sample_data


# Setup PyTorch

In [2]:
import torch

In [3]:
print(f"PyTorch Version: {torch.__version__}")
if (torch.cuda.is_available()):
    device = torch.device("cuda")
    print(f"Using GPU, CUDA version: {torch.version.cuda}, Number of GPUs: {torch.cuda.device_count()}")
else:
    device = torch.device("cpu")

PyTorch Version: 2.10.0+cu128
Using GPU, CUDA version: 12.8, Number of GPUs: 1


# Get a working tokenizer
Let's use OpenAI's tiktoken tokenizer - alghough I am actually interested in building a minimum tokenizer from scratch later.

In [4]:
!git clone https://github.com/openai/tiktoken.git

Cloning into 'tiktoken'...
remote: Enumerating objects: 504, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 504 (delta 3), reused 4 (delta 2), pack-reused 489 (from 2)
Receiving objects: 100% (504/504), 142.79 KiB | 2.75 MiB/s, done.
Resolving deltas: 100% (282/282), done.


In [5]:
import tiktoken

In [6]:
enc = tiktoken.get_encoding("r50k_base")

In [7]:
enc.encode("monkey d luffy")

[49572, 288, 300, 15352]

In [8]:
enc.decode(enc.encode("monkey d luffy"))

'monkey d luffy'

# Get the WebText dataset

In [9]:
from datasets import load_dataset

In [ ]:
openwebtext = load_dataset("openwebtext")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

plain_text/train-00000-of-00080.parquet:   0%|          | 0.00/303M [00:00<?, ?B/s]

...

plain_text/train-00076-of-00080.parquet:   0%|          | 0.00/303M [00:00<?, ?B/s]

plain_text/train-00077-of-00080.parquet:   0%|          | 0.00/303M [00:00<?, ?B/s]

plain_text/train-00078-of-00080.parquet:   0%|          | 0.00/301M [00:00<?, ?B/s]

plain_text/train-00079-of-00080.parquet:   0%|          | 0.00/303M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8013769 [00:00<?, ? examples/s]

In [ ]:
openwebtext

In [ ]:
openwebtext['train']

## Test tokenizing the training data

In [ ]:
enc.encode(openwebtext['train'][42]['text'])

# Embedding layer

In [ ]:
vocabulary_size = enc.n_vocab
d_model = 768

embedding_layer = torch.nn.Embedding(vocabulary_size, d_model)

Here we encode the phrase "monkey d luffy", which is then fed into the embedding_layer,
which produces d_model=768 embeddings, one 768 dimentional vector per token

In [ ]:
embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")]))

In [ ]:
embedding_layer.weight.shape

Here we can see that the embedding layer is rather simple - it is a n_vocab x d_model matrix. So the embedding process is simply taking w[encoding].

## Positional Encoding

In [ ]:
class LearnablePositionalEncoding(torch.nn.Module):
    def __init__(self, max_seq_len, d_model):
        super().__init__()
        self.pos_embeddings = torch.nn.Embedding(max_seq_len, d_model)
    def forward(self, x):
        # x is of shape (batch_size, seq_len, d_model)
        seq_len = x.size(1)

        position_ids = torch.arange(seq_len, dtype=torch.long, device=x.device)

        pos_enc = self.pos_embeddings(position_ids)
        return x + pos_enc


In [ ]:
pos_embed_layer = LearnablePositionalEncoding(1024, d_model)

In [ ]:
pos_embed_layer(embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")])))

# Decoder Only Transformer Layer

This is the key part of the transformer architecture. Each attention layer is made of multi-head attention, normalization, and position-wise feedforward.

In GPT-2, normalization is moved at the beginning of each layer.

In [ ]:
embedding = embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")]))

## Layer Normalization

https://arxiv.org/pdf/1607.06450

In [ ]:
layer_norm = torch.nn.LayerNorm(d_model)

In [ ]:
x = layer_norm(embedding)

In [ ]:
x

In [ ]:
x.shape

## Multi-Head Attention

In [ ]:
n_head = 12

# project into Q, K, V
projection = torch.nn.Linear(d_model, d_model * 3)

In [ ]:
projection(x).shape

In [ ]:
proj = projection(x).view(1, -1, 12, d_model // n_head * 3).permute(0, 2, 1, 3)

In [ ]:
q, k, v = proj.chunk(3, dim=3)

In [ ]:
import math
atten = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_model)

In [ ]:
mask = torch.tril(torch.ones_like(atten[0][0]))

In [ ]:
atten = atten.masked_fill(mask == 0, -float('inf'))

In [ ]:
atten

In [ ]:
softmax = torch.nn.Softmax(dim=-1)

atten = softmax(atten)

In [ ]:
atten

In [ ]:
torch.matmul(atten, v).transpose(1, 2).reshape(1, 4, d_model)

### Putting it together

In [ ]:
import torch.nn.functional as F
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        # project into K Q V
        self.input_linear = torch.nn.Linear(d_model, d_model * 3)
    def forward(self, x):
        batch_size, sequence_length, _ = x.size()
        proj = self.input_linear(x).view(batch_size, sequence_length, self.n_heads, self.d_model // self.n_heads * 3).permute(0, 2, 1, 3)
        q, k, v = proj.chunk(3, dim=-1)

        atten = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_model)
        mask = torch.tril(torch.ones_like(atten[0][0]))
        atten.masked_fill(mask == 0, -float('inf'))
        atten = F.softmax(atten, dim=-1)

        return torch.matmul(atten, v).transpose(1, 2).reshape(batch_size, sequence_length, d_model)


In [ ]:
multi_head_attention = MultiHeadAttention(d_model, n_head)
multi_head_attention(x)

## Position-wise Feed Forward

In [ ]:
d_hidden = 3072

In [ ]:
class DenseFeedForward(torch.nn.Module):
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.model = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(d_hidden, d_model)
        )
    def forward(self, x):
        return self.model(x)

In [ ]:
y = multi_head_attention(x)

In [ ]:
dff = DenseFeedForward(d_model, d_hidden)

In [ ]:
dff(y)

## The Full Transformer Layer

In [ ]:
class TransformerDecoderOnly(torch.nn.Module):
    def __init__(self, d_model, d_hidden, n_head):
        super().__init__()
        self.layer_norm_0 = torch.nn.LayerNorm(d_model)
        self.multi_head_attention = MultiHeadAttention(d_model, n_head)
        self.layer_norm_1 = torch.nn.LayerNorm(d_model)
        self.dff = DenseFeedForward(d_model, d_hidden)
    def forward(self, x):
        # layer normalization first
        x = self.layer_norm_0(x)

        # multi-head attention and residual
        identity = x
        x = self.multi_head_attention(x)
        x = x + identity

        # layer normalization before dense feed forward
        x = self.layer_norm_1(x)

        # dense feed forward and residual
        identity = x
        x = self.dff(x)
        return x + identity

In [ ]:
transformer_decoder_only = TransformerDecoderOnly(d_model, d_hidden, n_head)

transformer_decoder_only(embedding)

# Output Layer

The output layer is rather simple. matrix multiply with the input embedding matrix, and soft max. We can then reverse the tokenization.

In [ ]:
torch.matmul(transformer_decoder_only(embedding), embedding_layer.weight.transpose(0, 1))

In [ ]:
class Output(torch.nn.Module):
    def __init__(self, embedding_layer_weights):
        super().__init__()
        self.layer_norm = torch.nn.LayerNorm(embedding_layer_weights.shape[1])
        self.embedding_layer_weights = embedding_layer_weights

    def forward(self, x):
        # x is of shape (batch_size, seq_length, d_model)
        # self.embedding_layer_weights is of shape (vocab_size, d_model)
        x = self.layer_norm(x)
        return torch.matmul(x, self.embedding_layer_weights.transpose(0, 1))

# The full model

In [ ]:
class ToyGPT(torch.nn.Module):
    def __init__(self, d_model, d_hidden, n_head, vocab_size, layers, max_seq_len):
        super().__init__()
        self.embedding_layer = torch.nn.Embedding(vocabulary_size, d_model)
        self.pos_encoding_layer = LearnablePositionalEncoding(max_seq_len, d_model)
        self.transformers = torch.nn.Sequential()
        for i in range (0, layers):
            self.transformers.append(TransformerDecoderOnly(d_model, d_hidden, n_head))
        self.output_layer = Output(self.embedding_layer.weight)

    def forward(self, x):
        x = self.embedding_layer(x)
        x = self.pos_encoding_layer(x)
        x = self.transformers(x)
        return self.output_layer(x)


# Training

In [ ]:
batch_size = 48
max_seq_len = 1024

In [ ]:
enc._special_tokens

Pre-tokenize the training data to avoid CPU bottleneck.

In [ ]:
from datasets import Dataset

rows = openwebtext["train"].num_rows
bos_token = '<|endoftext|>'

def tokenize(examples):
    out = enc.encode(examples['text']+ bos_token, allowed_special={bos_token})
    return {'tokenized_text': out}

tokenized_data = openwebtext["train"].map(tokenize, remove_columns=openwebtext["train"].column_names, num_proc=32)


In [ ]:
def do_checkpoint(model, optimizer, running_encoding, row_number):
    torch.save({
        'row_number': row_number,
        'running_encoding': running_encoding,
        'optimizer_state_dict': optimizer.state_dict(),
        'model_state_dict': model.state_dict(),
    }, 'checkpoints/check_point_{}'.format(row_number))

In [ ]:
import gc
from tqdm.notebook import tqdm

def train(start_row_number = 0, check_point_interval = 100000, summary_writer = None):
    tokens_per_batch = (batch_size + 1) * max_seq_len // 2

    encoding = []

    model = ToyGPT(d_model, d_hidden, n_head, enc.n_vocab, 12, max_seq_len)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0006, weight_decay=0.01)
    model.to(device)
    model.train()

    model = torch.compile(model)

    if start_row_number != 0:
        checkpoint = torch.load('checkpoints/check_point_{}'.format(start_row_number))

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        encoding = checkpoint['running_encoding']

    write_summary_threshold = start_row_number
    check_point_threshold = (start_row_number + check_point_interval) // check_point_interval * check_point_interval

    data = tokenized_data.select(range(start_row_number, tokenized_data.num_rows)).to_iterable_dataset(num_shards=8)


    for row in enumerate(tqdm(data, total = rows, initial = start_row_number)):
        encoding.extend([enc._special_tokens[bos_token]] + row[1]['tokenized_text'])
        cur_row_number = start_row_number + row[0]
        if len(encoding) > tokens_per_batch:
            residual = encoding[tokens_per_batch:]
            encoding = encoding[0:tokens_per_batch + 1]
            batch = None
            target = None
            for i in range(0, batch_size):
              batch_row = torch.LongTensor(encoding[i * max_seq_len // 2 : i * max_seq_len // 2 + max_seq_len])
              batch_row = batch_row.to(device)
              target_row = torch.LongTensor(encoding[i * max_seq_len // 2 + 1 : i * max_seq_len // 2 + max_seq_len + 1])
              target_row = target_row.to(device)
              if batch is None:
                  batch = batch_row
                  target = target_row
              else:
                  batch = torch.cat((batch, batch_row), dim=0)
                  target = torch.cat((target, target_row), dim=0)

            batch = batch.to(device)
            batch = batch.reshape(batch_size, max_seq_len)
            target = target.to(device)
            target = target.reshape(batch_size, max_seq_len)

            optimizer.zero_grad()

            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                out = model(batch)
                loss = F.cross_entropy(out.view(-1, enc.n_vocab), target.view(-1))

            loss.backward()
            optimizer.step()

            encoding = residual
            if summary_writer and cur_row_number >= write_summary_threshold:
                write_summary_threshold = (write_summary_threshold + 1000) // 1000 * 1000
                summary_writer.add_scalar('Loss/train', loss.item(), cur_row_number)

            if (cur_row_number + 1 >= check_point_threshold):
                print("check pointing at {}".format(cur_row_number + 1))
                do_checkpoint(model, optimizer, encoding, cur_row_number + 1)
                check_point_threshold = (cur_row_number + 1 + check_point_interval) // check_point_interval * check_point_interval

            gc.collect()

In [ ]:
%load_ext tensorboard

In [ ]:
from torch.utils.tensorboard import SummaryWriter

summary_writer = SummaryWriter('./adamw_run_lr_0006')

In [ ]:
%tensorboard --logdir adamw_run_lr_0006


In [ ]:
train(start_row_number=0, check_point_interval=100000, summary_writer = summary_writer)